# 🛠️ Notebook 2 — Library Management: Full Implementation

We now **implement** the "best" design from notebook 1 and exercise it with realistic scenarios:

- Add books, copies, and members.
- **Search** by title / author / subject.
- **Borrow** and enforce loan limits.
- **Reserve** when no copy is free and auto-notify when one returns.
- **Return** with **fine** calculation.
- **Notify** overdue members.

Everything runs in this notebook with zero external dependencies (just `dataclasses` and `datetime`).


## 🛠️ Setup

```bash
cd 07-object-oriented-design/library-management
uv sync
```

In VS Code, pick the `.venv` kernel (top-right). If it doesn't appear: `Cmd+Shift+P` → **Reload Window**.


## 1. Domain classes

Small, frozen where possible, no behaviour beyond "what I am".


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import date, timedelta
from typing import Optional, Protocol
import itertools

@dataclass(frozen=True)
class Book:
    isbn: str
    title: str
    author: str
    subject: str

class BookItem:
    """A physical copy. Unique barcode, know its book and loan status."""
    _ids = itertools.count(1)
    def __init__(self, book: Book):
        self.barcode = f"BK-{next(BookItem._ids):04d}"
        self.book = book
        self.on_loan = False
    def __repr__(self):
        status = "OUT" if self.on_loan else "IN"
        return f"{self.barcode}:{self.book.title}({status})"

@dataclass
class Loan:
    member_id: str
    item: BookItem
    borrowed_on: date
    due_on: date
    returned_on: Optional[date] = None

@dataclass
class Reservation:
    member_id: str
    isbn: str
    created_on: date

@dataclass
class Member:
    id: str
    name: str
    email: str = ""
    active_loans: list[Loan] = field(default_factory=list)


## 2. Strategy — fine calculation

By hiding the fine rule behind a `Protocol`, we can swap policies (e.g., a discount for students) without touching `Library`.


In [ ]:
class FineCalculator(Protocol):
    def fine_for(self, loan: Loan, today: date) -> float: ...

class DailyFine:
    """$per_day for every day past due."""
    def __init__(self, per_day: float = 0.50):
        self.per_day = per_day
    def fine_for(self, loan: Loan, today: date) -> float:
        late = max(0, (today - loan.due_on).days)
        return round(late * self.per_day, 2)

class StudentDiscountFine:
    """Half price for members whose id starts with 'S-'."""
    def __init__(self, base: DailyFine):
        self.base = base
    def fine_for(self, loan: Loan, today: date) -> float:
        amt = self.base.fine_for(loan, today)
        return round(amt * 0.5, 2) if loan.member_id.startswith("S-") else amt


## 3. Observer — a real subscriber list

The previous section's `FineCalculator` is a **Strategy**: the library injects exactly one, calls
it, and *uses the number it returns*. Notifications are a different shape and deserve a different
pattern — several parties may care about the same event, and none of them return anything useful.

That is the **Observer** pattern, and it has three parts:

| Part | Here |
|---|---|
| **Observer interface** | `LibraryObserver.on_event(event, member_id, message)` |
| **Subject** — owns the list, publishes | `Library.subscribe()` / `Library._publish()` |
| **Concrete observers** | `PrintObserver`, `AuditLog`, `EmailObserver`… |

The tell-tale sign you have a real Observer and not a Strategy wearing its badge: you can
`subscribe` a **second** listener and both fire, and you can `unsubscribe` one at runtime.
`Library` never imports or names a concrete observer, so adding an SMS gateway tomorrow
touches zero existing lines.

In [ ]:
class LibraryObserver(Protocol):
    """React to something that happened. Returns nothing — this is a broadcast,
    not a question, which is exactly what separates Observer from Strategy."""
    def on_event(self, event: str, member_id: str, message: str) -> None: ...


class PrintObserver:
    """Stands in for email/push. Records what it sent so the demo can assert on it."""
    def __init__(self):
        self.sent: list[tuple[str, str]] = []
    def on_event(self, event: str, member_id: str, message: str) -> None:
        self.sent.append((member_id, message))
        print(f"[{event} -> {member_id}] {message}")


class AuditLog:
    """A SECOND observer, added later, with a completely different job.
    Library needed no change to support it — that is the payoff of the pattern."""
    def __init__(self):
        self.entries: list[tuple[str, str]] = []
    def on_event(self, event: str, member_id: str, message: str) -> None:
        self.entries.append((event, member_id))

## 4. Catalog — search

A tiny in-memory search. It stores books and returns matches by substring (case-insensitive). Separating *search* from *storage* lets you later plug in SQL, Elasticsearch, etc.


In [ ]:
class Catalog:
    def __init__(self):
        self._books: dict[str, Book] = {}

    def add(self, book: Book) -> None:
        self._books[book.isbn] = book

    def find(self, *, title: str = "", author: str = "", subject: str = "") -> list[Book]:
        def match(b: Book) -> bool:
            return (
                (not title   or title.lower()   in b.title.lower())   and
                (not author  or author.lower()  in b.author.lower())  and
                (not subject or subject.lower() in b.subject.lower())
            )
        return [b for b in self._books.values() if match(b)]

    def __contains__(self, isbn: str) -> bool:
        return isbn in self._books


## 5. The `Library` coordinator

The coordinator **orchestrates** — it doesn't compute fines itself, doesn't send emails itself. It applies *policy* (max loans, loan length) and delegates everything else.


In [ ]:
class Library:
    MAX_LOANS = 5
    LOAN_DAYS = 10

    def __init__(
        self,
        fine_calculator: FineCalculator | None = None,   # Strategy: exactly one
        observers: list[LibraryObserver] | None = None,  # Observer: zero to many
    ):
        self.catalog = Catalog()
        self.items_by_isbn: dict[str, list[BookItem]] = {}
        self.members: dict[str, Member] = {}
        self.reservations: dict[str, list[Reservation]] = {}  # isbn -> FIFO queue
        self.fine_calculator = fine_calculator or DailyFine()
        self._observers: list[LibraryObserver] = list(observers or [])

    # --- observer plumbing -----------------------------------------------
    def subscribe(self, observer: LibraryObserver) -> None:
        self._observers.append(observer)

    def unsubscribe(self, observer: LibraryObserver) -> None:
        self._observers.remove(observer)

    def _publish(self, event: str, member_id: str, message: str) -> None:
        """Fire-and-forget. Library does not know or care who is listening,
        and deliberately ignores whatever the observers return."""
        for obs in self._observers:
            obs.on_event(event, member_id, message)

    # --- admin ------------------------------------------------------------
    def add_book(self, book: Book, copies: int = 1) -> None:
        self.catalog.add(book)
        self.items_by_isbn.setdefault(book.isbn, [])
        for _ in range(copies):
            self.items_by_isbn[book.isbn].append(BookItem(book))

    def register(self, member: Member) -> None:
        self.members[member.id] = member

    # --- queries ----------------------------------------------------------
    def available_copies(self, isbn: str) -> int:
        return sum(1 for it in self.items_by_isbn.get(isbn, []) if not it.on_loan)

    # --- core use cases ---------------------------------------------------
    def borrow(self, member_id: str, isbn: str, today: date) -> Loan:
        member = self._member(member_id)
        if len(member.active_loans) >= self.MAX_LOANS:
            raise RuntimeError(f"{member.name} already has {self.MAX_LOANS} active loans")
        if isbn not in self.catalog:
            raise KeyError(f"unknown ISBN {isbn}")

        # Respect reservations: only the member at the head of the queue
        # can take the next freed copy.
        queue = self.reservations.get(isbn, [])
        if queue and queue[0].member_id != member_id:
            raise RuntimeError("book is reserved for another member")

        for item in self.items_by_isbn[isbn]:
            if not item.on_loan:
                item.on_loan = True
                loan = Loan(member_id, item, today, today + timedelta(days=self.LOAN_DAYS))
                member.active_loans.append(loan)
                if queue and queue[0].member_id == member_id:
                    queue.pop(0)  # fulfilled
                return loan
        raise RuntimeError("no copies available — try reserve()")

    def reserve(self, member_id: str, isbn: str, today: date) -> Reservation:
        self._member(member_id)
        if isbn not in self.catalog:
            raise KeyError(f"unknown ISBN {isbn}")
        if self.available_copies(isbn) > 0:
            raise RuntimeError("copies are available; just borrow")
        res = Reservation(member_id, isbn, today)
        self.reservations.setdefault(isbn, []).append(res)
        return res

    def return_item(self, loan: Loan, today: date) -> float:
        loan.item.on_loan = False
        loan.returned_on = today
        self.members[loan.member_id].active_loans.remove(loan)
        fine = self.fine_calculator.fine_for(loan, today)
        # Notify next reserver, if any.
        queue = self.reservations.get(loan.item.book.isbn, [])
        if queue:
            next_res = queue[0]
            self._publish(
                "reservation_ready",
                next_res.member_id,
                f'"{loan.item.book.title}" is now available — you can borrow it.',
            )
        return fine

    def notify_overdue(self, today: date) -> None:
        for member in self.members.values():
            for loan in member.active_loans:
                if today > loan.due_on:
                    days = (today - loan.due_on).days
                    self._publish(
                        "overdue",
                        member.id,
                        f'"{loan.item.book.title}" is {days} day(s) overdue.',
                    )

    # --- helpers ----------------------------------------------------------
    def _member(self, member_id: str) -> Member:
        if member_id not in self.members:
            raise KeyError(f"unknown member {member_id}")
        return self.members[member_id]

## 6. A realistic walkthrough

Let's run a full scenario: add a few books, register members, search, borrow up to the limit, reserve, return late with a fine, and see the notifier fire.


In [ ]:
# Two observers on one library: a notifier AND an audit log.
# Adding the second one required no change to Library — that is the test.
notifications = PrintObserver()
audit = AuditLog()
lib = Library(observers=[notifications, audit])

# --- catalog ---
clean_code = Book("978-0132350884", "Clean Code", "Robert C. Martin", "Programming")
ddd        = Book("978-0321125217", "Domain-Driven Design", "Eric Evans", "Programming")
sicp       = Book("978-0262510875", "SICP", "Abelson & Sussman", "Computer Science")

lib.add_book(clean_code, copies=2)
lib.add_book(ddd,        copies=1)
lib.add_book(sicp,       copies=1)

# --- members ---
ada  = Member("M-1", "Ada",  "ada@example.com")
ben  = Member("S-2", "Ben",  "ben@uni.edu")       # S- => student, gets discount later
lib.register(ada)
lib.register(ben)

# --- search ---
print("Search 'design':")
for b in lib.catalog.find(title="design"):
    print(" -", b.title, "by", b.author)

print("\nSearch author='Martin':")
for b in lib.catalog.find(author="Martin"):
    print(" -", b.title)

In [ ]:
today = date(2026, 4, 1)

loan1 = lib.borrow("M-1", clean_code.isbn, today)
loan2 = lib.borrow("S-2", clean_code.isbn, today)
print("After two borrows, available copies of Clean Code:", lib.available_copies(clean_code.isbn))

# Third borrow should fail — no copies left.
try:
    lib.borrow("M-1", clean_code.isbn, today)
except RuntimeError as e:
    print("Expected error:", e)

# Ben reserves instead.
res = lib.reserve("S-2", clean_code.isbn, today)
print("Ben's reservation:", res)


In [ ]:
# Ada returns 15 days later — 5 days overdue.
return_day = date(2026, 4, 16)
fine = lib.return_item(loan1, return_day)
print(f"Ada's fine: ${fine}")

# The return triggered a notification to Ben (next in the queue).
print("Notifications sent so far:", notifications.sent)
print("Audit log saw            :", audit.entries)
assert len(notifications.sent) == 1 and len(audit.entries) == 1, "both observers fired"

### Swap the fine policy (Strategy pattern in action)

Same `Library`, different rule. Students get 50% off.


In [ ]:
lib2 = Library(fine_calculator=StudentDiscountFine(DailyFine(per_day=0.50)))
lib2.add_book(clean_code, copies=1)

# ⚠️ Fresh Member object, NOT the `ben` from above. `Member.active_loans` is state
# that belongs to a (library, member) PAIR, but we hung it on Member — so sharing one
# Member object across two Library instances silently merges their loan counts and
# breaks MAX_LOANS. Real systems dodge this by keeping loans in the library's own
# ledger and looking them up by member id. Worth naming out loud in an interview.
ben2 = Member("S-2", "Ben", "ben@uni.edu")
lib2.register(ben2)  # S- prefix → student discount

loan = lib2.borrow("S-2", clean_code.isbn, date(2026, 4, 1))
fine = lib2.return_item(loan, date(2026, 4, 16))  # 5 days late
print(f"Ben's fine with student discount: ${fine}  (half of $2.50)")

### Overdue notifications

`notify_overdue(today)` sweeps all active loans and fires a notification per overdue item.


In [ ]:
lib3 = Library(observers=[PrintObserver()])
lib3.add_book(sicp, copies=1)
lib3.register(Member("M-1", "Ada"))   # fresh object, see the note above
loan = lib3.borrow("M-1", sicp.isbn, date(2026, 4, 1))   # due 2026-04-11
lib3.notify_overdue(date(2026, 4, 20))                    # 9 days late

## 7. Edge cases exercised

- ❌ **Unknown ISBN** raises `KeyError`.
- ❌ **Unknown member** raises `KeyError`.
- ❌ **Over max loans** raises `RuntimeError`.
- ❌ **Reserving when copies are free** raises `RuntimeError` (just borrow).
- ✅ Reservation queue is FIFO; only the head reserver can grab the next returned copy.
- ✅ Fine = 0 when returned on time.
- ✅ Max **5** active loans per member is enforced.

### 🕳️ A hole we are leaving open on purpose

`return_item` notifies the head of the reservation queue but never *expires* that claim.
If that member never comes back, the copy is blocked for everyone else forever. Real systems
attach a deadline to the reservation ("held for you until Friday") and pop it when it lapses.
Spotting this kind of missing-timeout question is worth real points in an interview — the
reservation exercise below asks you to close it.

In [ ]:
sanity = Library()
sanity.add_book(sicp, copies=1)
ada_s = Member("M-1", "Ada")          # fresh object per library — see the note above
sanity.register(ada_s)

assert sanity.available_copies(sicp.isbn) == 1

try:
    sanity.borrow("M-404", sicp.isbn, date(2026, 4, 1))
except KeyError as e:
    print("unknown member ok:", e)

try:
    sanity.reserve("M-1", sicp.isbn, date(2026, 4, 1))  # copies available
except RuntimeError as e:
    print("reserve-when-available ok:", e)

loan = sanity.borrow("M-1", sicp.isbn, date(2026, 4, 1))
# Returned 5 days early — no fine, and the copy goes back on the shelf.
assert sanity.return_item(loan, date(2026, 4, 6)) == 0.0
assert sanity.available_copies(sicp.isbn) == 1,     "returning frees the copy"
assert ada_s.active_loans == [],                    "returning clears the member's loans"

# --- The MAX_LOANS policy: claimed above, now actually proven -----------
limit_lib = Library()
limit_lib.register(Member("M-9", "Hoarder"))
for n in range(Library.MAX_LOANS + 1):              # one more title than allowed
    limit_lib.add_book(Book(f"isbn-{n}", f"Book {n}", "A", "S"), copies=1)
for n in range(Library.MAX_LOANS):
    limit_lib.borrow("M-9", f"isbn-{n}", date(2026, 4, 1))
try:
    limit_lib.borrow("M-9", f"isbn-{Library.MAX_LOANS}", date(2026, 4, 1))
    raise AssertionError("the 6th loan must be refused")
except RuntimeError as e:
    print("loan limit enforced ✅:", e)

# --- Reservation queue is FIFO and only the head may take the copy -------
q = Library()
q.add_book(sicp, copies=1)
for mid in ("M-1", "M-2", "M-3"):
    q.register(Member(mid, mid))
held = q.borrow("M-1", sicp.isbn, date(2026, 4, 1))   # last copy goes out
q.reserve("M-2", sicp.isbn, date(2026, 4, 2))         # M-2 queues first
q.reserve("M-3", sicp.isbn, date(2026, 4, 3))         # M-3 queues second
q.return_item(held, date(2026, 4, 5))
try:
    q.borrow("M-3", sicp.isbn, date(2026, 4, 5))       # queue-jumping
    raise AssertionError("M-3 must not jump ahead of M-2")
except RuntimeError as e:
    print("queue jumping blocked ✅:", e)
q.borrow("M-2", sicp.isbn, date(2026, 4, 5))          # the head reserver may
assert q.reservations[sicp.isbn] == [] or q.reservations[sicp.isbn][0].member_id == "M-3"

# --- Observer: many listeners, and unsubscribe actually detaches --------
o1, o2 = PrintObserver(), AuditLog()
q.subscribe(o1); q.subscribe(o2)
q.notify_overdue(date(2026, 5, 1))                    # M-2's loan is overdue
assert len(o1.sent) == 1 and len(o2.entries) == 1,   "both observers received the event"
q.unsubscribe(o2)
q.notify_overdue(date(2026, 5, 2))
assert len(o1.sent) == 2 and len(o2.entries) == 1,   "unsubscribe detaches one listener only"

print("all library invariants hold ✅")

## 8. Where to go next

- **Persistence**: swap the in-memory dicts for a SQLite-backed repository.
- **Concurrency**: two members clicking "borrow" at the same time — protect `items_by_isbn` with a lock, or move state into a DB transaction.
- **Richer search**: add pagination, subject taxonomies, full-text search.
- **Renewals**: allow extending a loan if no one is waiting in the reservation queue.
- **Role-based access**: separate `Librarian` from `Member`; only librarians can add/remove books.
- **Events**: replace the explicit `Notifier` call with an event bus so multiple subscribers can react.

The key takeaway: because each responsibility is in its own class, each of these extensions is a *small, local* change.
